# Per-category `min_ade` — KD arms on LCDrive val (n = 1000)

Every arm is scored **through the teacher's frozen action expert** (`StitchedAlpamayoR1`),
which is the head these arms were trained to drive. Scoring a student's own token head
instead gives a different — and for the CE-free arms, degenerate (`min_ade` 37.5239,
every generation malformed) — answer.

`min_ade` is **best-of-6** samples against ground truth, so it is an *oracle* over modes.
Lower is better. Comparisons are **paired per clip**, never a difference of aggregates.

Colour encodes magnitude **within each row**, so each category is judged on its own scale:
a category where every arm does badly does not paint the whole row dark.

In [1]:
import json, csv, numpy as np, pandas as pd
from pathlib import Path

TRAIN = Path("/data/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training")
SCENARIOS = Path("/data/datasets/physical_ai_av/lcdrive_physicalai_av_manifests"
                 "/lcdrive_val_primary_scenario_mysubset.csv")

# label -> stitched per-clip results file. Ordered ceiling -> floor.
ARMS = {
    "teacher":      "stitch_4b_teacher.json",
    "blockonly_e1": "stitch_4b_blockonly_checkpoint-1598.json",
    "blockonly_e2": "stitch_4b_blockonly_e3_checkpoint-3196.json",
    "blockonly_e3": "stitch_4b_blockonly_e3_checkpoint-4794.json",
    "kvonly_e1":    "stitch_4b_kvonly.json",
    "kvonly_e3":    "stitch_4b_kvonly_e3_checkpoint-4794.json",
    # ⚠️ NOT a distillation arm: the TEACHER with 8 expert layers bypassed (identity), the
    # causal test of the CKA prune candidates {18,19,24,25,27,32,33,34}. It belongs in this
    # table as an upper bound on any pruned-expert student -- a 28-layer expert distils
    # toward THIS ceiling, not the teacher's.
    "teacher_pruneB": "stitch_4b_teacher_pruneB.json",
    "kvband_e1":    "stitch_4b_kvband_checkpoint-1598.json",
    "ce":           "stitch_4b_ce.json",
}
ARMS = {k: v for k, v in ARMS.items() if (TRAIN / v).exists()}
print("arms found:", ", ".join(ARMS))

arms found: teacher, blockonly_e1, blockonly_e2, blockonly_e3, kvonly_e1, kvonly_e3, teacher_pruneB, kvband_e1, ce


In [2]:
# Each stitched JSON is a flat list of per-clip records carrying `clip_id`, which is what
# makes the join to scenario labels -- and the paired tests -- possible at all.
scores = {
    arm: {r["clip_id"]: r["min_ade"] for r in json.loads((TRAIN / f).read_text())}
    for arm, f in ARMS.items()
}
category = {r["clip_uuid"]: r["scenario_category"]
            for r in csv.DictReader(SCENARIOS.open())}

# Restrict to clips every arm scored, so all columns describe the SAME clip set.
clips = [c for c in category if all(c in s for s in scores.values())]
df = pd.DataFrame({arm: [scores[arm][c] for c in clips] for arm in ARMS},
                  index=pd.Index([category[c] for c in clips], name="category"))
print(f"{len(clips)} clips x {len(ARMS)} arms")

table = df.groupby("category").mean()
table.insert(0, "n", df.groupby("category").size())
table = table.sort_values("n", ascending=False)

overall = df.mean().to_frame().T
overall.insert(0, "n", len(clips))
overall.index = ["ALL"]
table = pd.concat([table, overall])

print(table.round(3).to_string())   # plain-text fallback
table.round(3)

1000 clips x 9 arms
                                   n  teacher  blockonly_e1  blockonly_e2  blockonly_e3  kvonly_e1  kvonly_e3  teacher_pruneB  kvband_e1      ce
General Training/Validation      341    0.385         1.511         1.294         1.296      1.871      1.670           0.716      2.644   8.957
Lane Keeping Curve                59    0.976         4.300         3.491         3.644      6.574      5.773           1.984      6.032   8.380
Lead Vehicle Following            56    0.625         2.001         1.637         1.475      2.567      2.483           0.830      3.446   9.384
Nudge Static Obstacle Maneuver    56    0.482         1.669         1.469         1.372      1.911      1.725           0.685      2.314   4.529
Lane Keeping                      53    0.554         1.837         1.493         1.403      2.343      2.086           0.818      2.761   4.030
Nudge Maneuver                    53    0.602         1.775         1.479         1.417      1.974      1.812 

,n,teacher,blockonly_e1,blockonly_e2,blockonly_e3,kvonly_e1,kvonly_e3,teacher_pruneB,kvband_e1,ce
General Training/Validation,341,0.385,1.511,1.294,1.296,1.871,1.670,0.716,2.644,8.957
Lane Keeping Curve,59,0.976,4.300,3.491,3.644,6.574,5.773,1.984,6.032,8.380
Lead Vehicle Following,56,0.625,2.001,1.637,1.475,2.567,2.483,0.830,3.446,9.384
Nudge Static Obstacle Maneuver,56,0.482,1.669,1.469,1.372,1.911,1.725,0.685,2.314,4.529
Lane Keeping,53,0.554,1.837,1.493,1.403,2.343,2.086,0.818,2.761,4.030
Nudge Maneuver,53,0.602,1.775,1.479,1.417,1.974,1.812,0.793,2.207,2.963
Speed Control,53,0.597,3.297,2.805,2.667,3.302,3.160,0.996,4.171,6.575
Stop for Vehicle,48,0.447,1.490,1.294,1.218,2.980,2.822,0.682,3.058,3.120
Vulnerable Road Users (VRU),47,0.632,1.401,1.356,1.319,1.414,1.330,0.588,1.725,2.337
Lane Change,47,0.880,2.840,2.340,2.260,3.416,3.113,1.638,4.253,12.459


In [3]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential = ONE hue, light -> dark (never a rainbow, never red/green: both fail CVD).
# Light = low min_ade = better.
SEQ = LinearSegmentedColormap.from_list("seq", ["#f2f7fb", "#c8dbeb", "#7fa9cd", "#3d6f9e", "#1b3d5c"])

arm_cols = [c for c in table.columns if c != "n"]

styled = (
    table.style
    # axis=1 -> normalise WITHIN each row, so every category is judged on its own scale.
    .background_gradient(cmap=SEQ, subset=arm_cols, axis=1, text_color_threshold=0.45)
    .format({"n": "{:.0f}", **{c: "{:.3f}" for c in arm_cols}})
    .set_caption("min_ade through the teacher's expert — lower is better; "
                 "colour is row-relative (per category)")
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                          ("padding-bottom", "8px"), ("color", "#444")]},
        {"selector": "th", "props": [("font-weight", "600"), ("text-align", "right")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)
styled

,n,teacher,blockonly_e1,blockonly_e2,blockonly_e3,kvonly_e1,kvonly_e3,teacher_pruneB,kvband_e1,ce
General Training/Validation,341,0.385,1.511,1.294,1.296,1.871,1.670,0.716,2.644,8.957
Lane Keeping Curve,59,0.976,4.300,3.491,3.644,6.574,5.773,1.984,6.032,8.380
Lead Vehicle Following,56,0.625,2.001,1.637,1.475,2.567,2.483,0.830,3.446,9.384
Nudge Static Obstacle Maneuver,56,0.482,1.669,1.469,1.372,1.911,1.725,0.685,2.314,4.529
Lane Keeping,53,0.554,1.837,1.493,1.403,2.343,2.086,0.818,2.761,4.030
Nudge Maneuver,53,0.602,1.775,1.479,1.417,1.974,1.812,0.793,2.207,2.963
Speed Control,53,0.597,3.297,2.805,2.667,3.302,3.160,0.996,4.171,6.575
Stop for Vehicle,48,0.447,1.490,1.294,1.218,2.980,2.822,0.682,3.058,3.120
Vulnerable Road Users (VRU),47,0.632,1.401,1.356,1.319,1.414,1.330,0.588,1.725,2.337
Lane Change,47,0.880,2.840,2.340,2.260,3.416,3.113,1.638,4.253,12.459


## Paired comparison: `blockonly` − `kvonly_e3`, within category

Negative favours `blockonly`. `z` is a paired one-sample statistic on the per-clip
differences, so it accounts for the fact that some categories are simply harder.

Colour here is **diverging** — two hues with a neutral midpoint at exactly 0 — because the
quantity has polarity (which arm wins), not magnitude.

In [4]:
# Matched budget: both arms at 3 epochs, same init and schedule.
a, b = "blockonly_e3", "kvonly_e3"
rows = []
for cat, g in df.groupby("category"):
    d = (g[a] - g[b]).to_numpy()
    z = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 and d.std() > 0 else np.nan
    rows.append({"category": cat, "n": len(d), f"{a}": g[a].mean(),
                 f"{b}": g[b].mean(), "delta": d.mean(), "z": z})
d_all = (df[a] - df[b]).to_numpy()
rows.append({"category": "ALL", "n": len(d_all), f"{a}": df[a].mean(), f"{b}": df[b].mean(),
             "delta": d_all.mean(),
             "z": d_all.mean() / (d_all.std(ddof=1) / np.sqrt(len(d_all)))})
paired = pd.DataFrame(rows).set_index("category")

# Diverging: two hues + a NEUTRAL GREY midpoint, symmetric about 0 so the midpoint is
# genuinely "no difference" rather than wherever the data happens to centre.
DIV = LinearSegmentedColormap.from_list(
    "div", ["#1b3d5c", "#7fa9cd", "#eceff1", "#e0a367", "#8a4b12"])
lim = float(np.nanmax(np.abs(paired["delta"])))

styled_paired = (paired.style
   .background_gradient(cmap=DIV, subset=["delta"], vmin=-lim, vmax=lim, text_color_threshold=0.45)
   .format({"n": "{:.0f}", a: "{:.3f}", b: "{:.3f}", "delta": "{:+.3f}", "z": "{:+.2f}"})
   .set_caption(f"{a} − {b} per category · negative favours {a} · |z| > 2 ≈ significant")
   .set_table_styles([
       {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"),
                                         ("padding-bottom", "8px"), ("color", "#444")]},
       {"selector": "th.row_heading", "props": [("text-align", "left")]},
       {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
   ]))

print(paired.round(3).to_string())   # plain-text fallback
styled_paired

                                   n  blockonly_e3  kvonly_e3  delta       z
category                                                                    
Cut-In                            36         1.428      1.606 -0.177  -1.202
General Training/Validation      341         1.296      1.670 -0.374  -4.150
Intersection Navigation           42         1.258      1.762 -0.504  -2.371
Lane Change                       47         2.260      3.113 -0.853  -2.560
Lane Keeping                      53         1.403      2.086 -0.683  -2.988
Lane Keeping Curve                59         3.644      5.773 -2.129  -4.971
Lead Vehicle Following            56         1.475      2.483 -1.008  -4.114
Merging                           44         1.796      1.892 -0.096  -0.826
Nudge Maneuver                    53         1.417      1.812 -0.395  -2.676
Nudge Static Obstacle Maneuver    56         1.372      1.725 -0.353  -2.275
Speed Control                     53         2.667      3.160 -0.493  -3.045

,n,blockonly_e3,kvonly_e3,delta,z
category,,,,,
Cut-In,36,1.428,1.606,-0.177,-1.20
General Training/Validation,341,1.296,1.670,-0.374,-4.15
Intersection Navigation,42,1.258,1.762,-0.504,-2.37
Lane Change,47,2.260,3.113,-0.853,-2.56
Lane Keeping,53,1.403,2.086,-0.683,-2.99
Lane Keeping Curve,59,3.644,5.773,-2.129,-4.97
Lead Vehicle Following,56,1.475,2.483,-1.008,-4.11
Merging,44,1.796,1.892,-0.096,-0.83
Nudge Maneuver,53,1.417,1.812,-0.395,-2.68
